# Create MERlin data organization

In [ ]:
import os
import sys
from pathlib import Path
import pandas as pd

MERCI_DIR    = Path(os.getcwd()).parent.parent.parent   # MERci/ (notebook lives in MERci/notebooks/prepare_imaging/<variant>/)
SAMPLE_DIR   = MERCI_DIR.parent                  # experiment root, e.g. LT027_saving_time/
METADATA_DIR = SAMPLE_DIR / "metadata"
SETTINGS_DIR = SAMPLE_DIR / "settings"
sys.path.insert(0, str(MERCI_DIR / "src"))

from MERci.common.experiment_info import resolve_sample_identity
from MERci.acquisition.data_organization import create_data_organization
from MERci.acquisition.dave import annotate_dave_with_round_info

MICROSCOPE  = "MF3"

# SAMPLE_NAME is the TRUE top-level experiment id (e.g. "LT058_sample_07") and
# IMAGING_DIR is the acquisition-type subfolder (e.g. "merfish", "" if flat),
# both auto-detected from the folder structure -- NOT SAMPLE_DIR.name, which
# is only this acquisition's own local folder name once split into sibling
# acquisition-type subfolders.
SAMPLE_NAME, IMAGING_DIR = resolve_sample_identity(MERCI_DIR)

print(f"SAMPLE_DIR   : {SAMPLE_DIR}")
print(f"SAMPLE_NAME  : {SAMPLE_NAME}")
print(f"METADATA_DIR : {METADATA_DIR}")
print(f"SETTINGS_DIR : {SETTINGS_DIR}")

## Auto-detect frame tables

Lists all `frame-table-*.csv` files in the metadata folder. The bits and cells
frame tables are selected by the kind token in the filename
(`frame-table-bits-*` / `frame-table-cells-*`); transit tables are ignored.
Override `BITS_FT` / `CELLS_FT` manually if needed.

In [ ]:
ft_files = sorted(
    METADATA_DIR.glob("frame-table-*.csv"),
    key=lambda p: p.stat().st_mtime, reverse=True
)

print("Frame tables found (newest first):")
for f in ft_files:
    ft = pd.read_csv(f, index_col=0)
    colors = sorted(ft["color"].dropna().unique().astype(int))
    print(f"  {f.name}  →  colors: {colors}")

# Select by the kind token in the filename (frame-table-<kind>-<name>.csv);
# transit frame tables (all-blank) are ignored.
BITS_FT  = next(f for f in ft_files if f.name.startswith("frame-table-bits-"))
CELLS_FT = next(f for f in ft_files if f.name.startswith("frame-table-cells-"))

print(f"\nBits  frame table : {BITS_FT.name}")
print(f"Cells frame table : {CELLS_FT.name}")

## Round – bit – color mapping

The round → bit → colour mapping is now defined in **notebook 03** (which derives
`N_HYBS` from it) and saved to `round_bit_color_map.csv`. Here it is read back for the
data organization and the Dave annotation. Re-run notebook 03 to change the mapping.

In [ ]:
# round_bit_color is defined & saved in notebook 03; read it back here.
rbc_path = METADATA_DIR / "round_bit_color_map.csv"
if not rbc_path.exists():
    raise FileNotFoundError(
        f"{rbc_path} not found — run notebook 03 first "
        f"(it now defines the round–bit–color mapping and N_HYBS)."
    )
rbc_df          = pd.read_csv(rbc_path)
round_bit_color = [(int(r), int(b), int(c))
                   for r, b, c in rbc_df[["round", "bit", "color"]].itertuples(index=False, name=None)]
print(f"Loaded {len(round_bit_color)} (round, bit, color) rows from {rbc_path.name}")
print(rbc_df.to_string(index=False))

## Series patterns from round_info.csv

Reads the bits and cells series patterns so the correct `imageType` and
`imageRegExp` fields are written into the data organization.  
Re-run notebook 04 first if `round_info.csv` is out of date.

In [ ]:
round_info_path = METADATA_DIR / "round_info.csv"
round_info      = pd.read_csv(round_info_path)

# Multi-boundary round_info carries per-segment rows (imaging_type in
# {cells, bits, transit} + a 'segment' column). Pick the bits/cells series by
# imaging_type so transit movies are never selected; fall back to name matching
# for the legacy schema.
MULTI_BOUNDARY = "segment" in round_info.columns
if "imaging_type" in round_info.columns:
    itype        = round_info["imaging_type"].astype(str).str.lower()
    bits_rows    = round_info[itype == "bits"]
    cells_rows   = round_info[itype == "cells"]
else:
    bits_rows    = round_info[~round_info["series"].str.contains("cells")]
    cells_rows   = round_info[ round_info["series"].str.contains("cells")]

bits_series  = bits_rows.iloc[0]["series"]
cells_series = cells_rows.iloc[0]["series"]

print(f"Multi-boundary layout : {MULTI_BOUNDARY}")
print(f"Bits  series (sample) : {bits_series}")
print(f"Cells series (sample) : {cells_series}")

if MULTI_BOUNDARY:
    tissues = sorted(round_info["tissue"].dropna().astype(int).unique())
    print(f"\nTissues present       : {tissues}")
    print("NOTE: multi-tissue MERlin analysis is per tissue, and each boundary is a\n"
          "distinct movie (imageType). The cell below builds ONE data-organization from\n"
          "the representative series above; for a per-tissue / per-boundary MERlin run,\n"
          "confirm the intended workflow before relying on this file.")

## Build and save data organization

In [ ]:
readouts  = pd.read_csv(MERCI_DIR / "data" / "readouts.csv")
ft_bits   = pd.read_csv(BITS_FT,  index_col=0)
ft_cells  = pd.read_csv(CELLS_FT, index_col=0)

data_org = create_data_organization(
    bits_frame_table  = ft_bits,
    cells_frame_table = ft_cells,
    round_bit_color   = round_bit_color,
    readouts          = readouts,
    bits_series       = bits_series,
    cells_series      = cells_series,
    include_dapi      = True,
)

out_name = f"data_organization_{MICROSCOPE.upper()}_{SAMPLE_NAME}.csv"
out_path = METADATA_DIR / out_name
data_org.to_csv(out_path, index=False)
print(f"Saved: {out_path}")
print(f"\n{len(data_org)} rows  ({len(data_org)-1} bits + DAPI)")
data_org

## Annotate Dave XML with bit information

Adds per-round XML comments to the Dave config generated by notebook 04.
Imaging round 1 is the cells acquisition (no bits), so the bit comments attach to
the fluidics loops that precede each bits imaging round (rounds 2…N+1). The
hyb/bit-indexed `round_bit_color` is shifted `+1` here to match those imaging-round
numbers.

Re-run this cell whenever the round–bit–color mapping changes.

In [ ]:
from MERci.acquisition.dave import dave_config_filename

# Construct the exact expected filename (MICROSCOPE + N_HYBS, from notebook 03's
# round_bit_color_map.csv, read above) rather than globbing settings/dave-*.xml
# and guessing which match is "the" recipe -- that guess breaks as soon as more
# than one dave-*.xml lives in this settings/ folder (e.g. two acquisitions
# sharing one sample folder), since alphabetical sort picks "…-13hybs-…" before
# "…-9hybs-…" (string comparison, not numeric).
N_HYBS    = int(rbc_df["round"].max())
dave_path = SETTINGS_DIR / dave_config_filename(MICROSCOPE, N_HYBS, SAMPLE_NAME)

if not dave_path.exists():
    print(f"No {dave_path.name} found in settings/ — run notebook 04 first.")
else:
    print(f"Annotating: {dave_path.name}")
    # round_bit_color is hyb/bit-indexed (1..N) for data-organization, but the
    # Dave recipe images bits in imaging rounds 2..N+1 (round 1 = cells), so the
    # annotation round indices are shifted +1 to line up with the Fluidics loops.
    annotate_rbc = [(r + 1, bit, color) for (r, bit, color) in round_bit_color]
    annotate_dave_with_round_info(dave_path, annotate_rbc)
    print("Done. Preview of annotated file:")
    with open(dave_path, encoding="ISO-8859-1") as fh:
        print(fh.read())